# Day 15：大模型底层原理、Token 分词与企业 API 成本核算 (`v0.2.0-day15`)

> **🎙️ 讲师旁白**：欢迎来到《智枢企业级智能体平台》第二阶段第 1 天！今天我们将从原理到代码，掌握 **NLP → Transformer → Pretrain/SFT/RLHF** 的核心机制，并实战编写与验证 `project_iterations/day_15` 中的多语言 Token 估算器与 5 大主流模型成本核算引擎。

## 1. 动手理解 Transformer Self-Attention（缩放点积注意力）
Transformer 的核心公式为 $\text{Attention}(Q, K, V) = \text{softmax}(\frac{QK^T}{\sqrt{d_k}})V$。下面用纯 Python 演示一个极简的注意力权重计算过程。

In [1]:
import math

def dot(a: list[float], b: list[float]) -> float:
    return sum(x * y for x, y in zip(a, b))

def softmax(scores: list[float]) -> list[float]:
    max_s = max(scores)
    exps = [math.exp(s - max_s) for s in scores]
    total = sum(exps)
    return [e / total for e in exps]

# 假设当前词 Query 与三个历史词 Key 的向量
query = [1.0, 0.8]
keys = [[1.0, 0.9], [0.1, 0.2], [-0.5, 0.1]]
tokens = ["智枢", "智能体", "昨天"]
d_k = len(query)

raw_scores = [dot(query, k) / math.sqrt(d_k) for k in keys]
weights = softmax(raw_scores)

for tok, w in zip(tokens, weights):
    print(f"Token '{tok}' 获得的注意力权重: {w:.4f}")

Token '智枢' 获得的注意力权重: 0.6344
Token '智能体' 获得的注意力权重: 0.2259
Token '昨天' 获得的注意力权重: 0.1397


## 2. 导入 `project_iterations/day_15` 并验证中英文 Token 估算差异
在通用 BPE 分词器中，CJK 中文字符的 Token 密度显著高于 ASCII 英文字符。我们来实测 `estimate_tokens`。

In [2]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day15_dir = repo_root / "project_iterations" / "day_15"
if str(day15_dir) not in sys.path:
    sys.path.insert(0, str(day15_dir))

from zhishu_agent import MODEL_PRICING, compare_models_cost, estimate_api_cost, estimate_tokens

samples = [
    ("纯中文", "智枢企业级智能体平台正在进行第二阶段研发"),
    ("纯英文", "ZhiShu Enterprise AI Agent Platform Stage 2"),
    ("中英混合", "使用 DeepSeek-V3 和 GPT-4o 进行 API 成本对比"),
]
for label, text in samples:
    print(f"[{label}] 字符数={len(text):>2} | 估算Token={estimate_tokens(text):>2} | 文本: {text}")

[纯中文] 字符数=20 | 估算Token=30 | 文本: 智枢企业级智能体平台正在进行第二阶段研发
[纯英文] 字符数=43 | 估算Token=10 | 文本: ZhiShu Enterprise AI Agent Platform Stage 2
[中英混合] 字符数=35 | 估算Token=20 | 文本: 使用 DeepSeek-V3 和 GPT-4o 进行 API 成本对比


## 3. 单模型 API 调用成本核算（`estimate_api_cost`）
查看 `GPT-4o` 在一段典型企业问答场景下的 Token 明细与 USD / RMB 双币种费用。

In [3]:
prompt = "请总结智枢平台第一阶段与第二阶段的核心演进路线。"
completion = "第一阶段夯实了 Python 工程基础与 CLI 多轮对话助手；第二阶段将引入大模型原理、FastAPI 与 SSE 流式服务。"

gpt4o_cost = estimate_api_cost("GPT-4o", prompt, completion)
for k, v in gpt4o_cost.items():
    print(f"{k}: {v}")

model: GPT-4o
provider: OpenAI
prompt_tokens: 36
completion_tokens: 63
total_tokens: 99
input_cost_usd: 9e-05
output_cost_usd: 0.00063
total_cost_usd: 0.00072
input_cost_rmb: 0.0006525
output_cost_rmb: 0.0045675
total_cost_rmb: 0.00522


## 4. 5 大主流模型横向比价与企业月度预算模拟（`compare_models_cost`）
模拟企业客服每天 10,000 次调用（每月 300,000 次）时的费用差距，直观感受模型分级路由的价值。

In [4]:
monthly_calls = 300_000
results = compare_models_cost(prompt, completion)

print(f"{'模型':<18} {'总Token':>7} {'单次USD($)':>12} {'单次RMB(¥)':>12} {'月度预算(RMB)':>14}")
print("-" * 68)
for item in results:
    monthly_rmb = item["total_cost_rmb"] * monthly_calls
    print(
        f"{item['model']:<18} {item['total_tokens']:>7} "
        f"{item['total_cost_usd']:>12.6f} {item['total_cost_rmb']:>12.6f} "
        f"¥{monthly_rmb:>13,.2f}"
    )

模型                  总Token     单次USD($)     单次RMB(¥)      月度预算(RMB)
--------------------------------------------------------------------
Llama-3.1-70B           99     0.000071     0.000515 ¥       154.45
DeepSeek-V3             99     0.000079     0.000573 ¥       171.87
Qwen-Plus               99     0.000090     0.000652 ¥       195.75
GPT-4o                  99     0.000720     0.005220 ¥     1,566.00
Claude-3.5-Sonnet       99     0.001053     0.007634 ¥     2,290.28
